# Controlled K-Means preprocessing experiment

## 1. Experiment objective

Compare three preprocessing strategies under the same K-Means configuration. This notebook reads completed experiment results; it does not select a final strategy.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
project_root = Path.cwd()
if not (project_root / 'data').exists():
    project_root = project_root.parent
experiment_dir = project_root / 'data' / 'experiments'
runs = pd.read_csv(experiment_dir / 'kmeans_preprocessing_comparison.csv')
summary = pd.read_csv(experiment_dir / 'kmeans_preprocessing_summary.csv')

## 2. Input feature set and preprocessing

CustomerID is excluded. K-Means uses Recency, Frequency, MonetaryValue, UniqueProducts, and CustomerLifetimeDays. The compared inputs are standard, log-standard, and log-robust.

In [ ]:
print(f'Run rows: {len(runs)}')
print(f'Summary rows: {len(summary)}')
display(summary.round(4))

## 3. Evaluation metrics

Higher Silhouette and Calinski-Harabasz scores are preferred; lower Davies-Bouldin is preferred. Mean pairwise ARI measures assignment stability across random seeds, with values near 1 indicating close agreement.

## 4. Results by k and visual comparison

In [ ]:
metric_specs = [
    ('mean_silhouette', 'Mean Silhouette', True),
    ('mean_davies_bouldin', 'Mean Davies-Bouldin', False),
    ('mean_calinski_harabasz', 'Mean Calinski-Harabasz', True),
]
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for axis, (column, title, _) in zip(axes, metric_specs):
    for strategy, group in summary.groupby('preprocessing_strategy'):
        axis.plot(group['k'], group[column], marker='o', label=strategy)
    axis.set_title(title)
    axis.set_xlabel('Number of clusters (k)')
    axis.set_xticks(range(2, 9))
axes[0].set_ylabel('Score')
axes[-1].legend(title='Preprocessing')
fig.suptitle('K-Means internal metrics by preprocessing strategy')
fig.tight_layout()
plt.show()

## 5. Stability analysis

In [ ]:
stability = summary.pivot(index='k', columns='preprocessing_strategy', values='mean_pairwise_ari')
display(stability.round(4))
stability.plot(marker='o', figsize=(9, 5))
plt.ylim(0.90, 1.005)
plt.ylabel('Mean pairwise Adjusted Rand Index')
plt.title('Assignment stability across five random seeds')
plt.tight_layout()
plt.show()

## 6. Cluster balance analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for strategy, group in summary.groupby('preprocessing_strategy'):
    axes[0].plot(group['k'], group['average_smallest_cluster_percentage'], marker='o', label=strategy)
    axes[1].plot(group['k'], group['average_largest_cluster_percentage'], marker='o', label=strategy)
axes[0].set_title('Average smallest cluster share')
axes[1].set_title('Average largest cluster share')
for axis in axes:
    axis.set_xlabel('Number of clusters (k)')
    axis.set_ylabel('Customers (%)')
    axis.set_xticks(range(2, 9))
axes[1].legend(title='Preprocessing')
fig.tight_layout()
plt.show()

In [ ]:
balance_columns = [
    'preprocessing_strategy', 'k',
    'average_smallest_cluster_percentage',
    'average_largest_cluster_percentage',
]
display(summary[balance_columns].round(3))

## 7. Preliminary interpretation

- Log-standard k=2 combines high stability, balanced sizes, and the strongest internal metrics among transformed inputs.
- Log-robust k=2 is exactly stable and similarly balanced, with slightly weaker internal metrics.
- Standard k=6 or k=7 has competitive internal metrics but isolates extremely small clusters, which may mainly capture retained outliers.
- These candidates require customer-profile interpretation before any winner is selected.